# AI-Based MBA Student Placement Prediction System

### Predicting MBA Student Placement Using Supervised Machine Learning

## Algorithms Used
- Logistic Regression
- Decision Tree
- Random Forest
- Support Vector Machine (SVM)

**Final Selected Model:** Logistic Regression

**Programming Language:** Python

**Libraries Used:** Pandas, NumPy, Matplotlib, Scikit-learn, Joblib

## Problem Statement

The objective of this project is to predict whether a student will be placed based on academic performance, work experience, and employability test scores using supervised machine learning techniques. Four classification algorithms are trained, evaluated, and compared to identify the best-performing model for deployment.

## Import Libraries

In [238]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib
import os

# Preprocessing
from sklearn.preprocessing import OneHotEncoder

# Model Selection
from sklearn.model_selection import train_test_split

# Models
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler

# Evaluation
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_auc_score
)

## Load Dataset

In [239]:
df = pd.read_csv(r"../dataset/Placement_Data_Full_Class.csv")

## Dataset Overview

In [240]:
df.head()

,student_id,gender,ssc_percentage,ssc_board,hsc_percentage,hsc_board,hsc_stream,degree_percentage,degree_type,work_experience,employability_test_percentage,specialisation,mba_percentage,status,salary
0,1,M,67.00,Others,91.00,Others,Commerce,58.00,Sci&Tech,No,55.0,Mkt&HR,58.80,Placed,270000.0
1,2,M,79.33,Central,78.33,Others,Science,77.48,Sci&Tech,Yes,86.5,Mkt&Fin,66.28,Placed,200000.0
2,3,M,65.00,Central,68.00,Central,Arts,64.00,Comm&Mgmt,No,75.0,Mkt&Fin,57.80,Placed,250000.0
3,4,M,56.00,Central,52.00,Central,Science,52.00,Sci&Tech,No,66.0,Mkt&HR,59.43,Not Placed,NaN
4,5,M,85.80,Central,73.60,Central,Commerce,73.30,Comm&Mgmt,No,96.8,Mkt&Fin,55.50,Placed,425000.0


In [241]:
df.shape

(215, 15)

In [242]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 215 entries, 0 to 214
Data columns (total 15 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   student_id                     215 non-null    int64  
 1   gender                         215 non-null    object 
 2   ssc_percentage                 215 non-null    float64
 3   ssc_board                      215 non-null    object 
 4   hsc_percentage                 215 non-null    float64
 5   hsc_board                      215 non-null    object 
 6   hsc_stream                     215 non-null    object 
 7   degree_percentage              215 non-null    float64
 8   degree_type                    215 non-null    object 
 9   work_experience                215 non-null    object 
 10  employability_test_percentage  215 non-null    float64
 11  specialisation                 215 non-null    object 
 12  mba_percentage                 215 non-null    flo

In [243]:
df.describe()

,student_id,ssc_percentage,hsc_percentage,degree_percentage,employability_test_percentage,mba_percentage,salary
count,215.000000,215.000000,215.000000,215.000000,215.000000,215.000000,148.000000
mean,108.000000,67.303395,66.333163,66.370186,72.100558,62.278186,288655.405405
std,62.209324,10.827205,10.897509,7.358743,13.275956,5.833385,93457.452420
min,1.000000,40.890000,37.000000,50.000000,50.000000,51.210000,200000.000000
25%,54.500000,60.600000,60.900000,61.000000,60.000000,57.945000,240000.000000
50%,108.000000,67.000000,65.000000,66.000000,71.000000,62.000000,265000.000000
75%,161.500000,75.700000,73.000000,72.000000,83.500000,66.255000,300000.000000
max,215.000000,89.400000,97.700000,91.000000,98.000000,77.890000,940000.000000


## Missing Value Analysis

In [244]:
df.isnull().sum()

student_id                        0
gender                            0
ssc_percentage                    0
ssc_board                         0
hsc_percentage                    0
hsc_board                         0
hsc_stream                        0
degree_percentage                 0
degree_type                       0
work_experience                   0
employability_test_percentage     0
specialisation                    0
mba_percentage                    0
status                            0
salary                           67
dtype: int64

## Data Cleaning
The `salary` column contains missing values for students who were not placed. Since salary is not available for all records and is not required for predicting placement status, it is removed from the dataset.

In [245]:
df=df.drop('salary',axis=1)

In [246]:
df.columns

Index(['student_id', 'gender', 'ssc_percentage', 'ssc_board', 'hsc_percentage',
       'hsc_board', 'hsc_stream', 'degree_percentage', 'degree_type',
       'work_experience', 'employability_test_percentage', 'specialisation',
       'mba_percentage', 'status'],
      dtype='object')

In [247]:
df.dtypes

student_id                         int64
gender                            object
ssc_percentage                   float64
ssc_board                         object
hsc_percentage                   float64
hsc_board                         object
hsc_stream                        object
degree_percentage                float64
degree_type                       object
work_experience                   object
employability_test_percentage    float64
specialisation                    object
mba_percentage                   float64
status                            object
dtype: object

## Feature Encoding
Categorical features are converted into numerical values using One-Hot Encoding. The first category is dropped to avoid multicollinearity (dummy variable trap).

In [248]:
encoder=OneHotEncoder(drop="first",sparse_output=False)
categorical_columns=['gender','ssc_board','hsc_board','hsc_stream','degree_type','work_experience','specialisation']
encoded=encoder.fit_transform(df[categorical_columns])
new_dataset=pd.DataFrame(encoded,columns=encoder.get_feature_names_out(categorical_columns),index=df.index)
df=df.drop(categorical_columns,axis=1)
df=pd.concat([df,new_dataset],axis=1)

In [249]:
df.head()

,student_id,ssc_percentage,hsc_percentage,degree_percentage,employability_test_percentage,mba_percentage,status,gender_M,ssc_board_Others,hsc_board_Others,hsc_stream_Commerce,hsc_stream_Science,degree_type_Others,degree_type_Sci&Tech,work_experience_Yes,specialisation_Mkt&HR
0,1,67.00,91.00,58.00,55.0,58.80,Placed,1.0,1.0,1.0,1.0,0.0,0.0,1.0,0.0,1.0
1,2,79.33,78.33,77.48,86.5,66.28,Placed,1.0,0.0,1.0,0.0,1.0,0.0,1.0,1.0,0.0
2,3,65.00,68.00,64.00,75.0,57.80,Placed,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,4,56.00,52.00,52.00,66.0,59.43,Not Placed,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0
4,5,85.80,73.60,73.30,96.8,55.50,Placed,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0


In [250]:
df.shape

(215, 16)

In [251]:
df["status_Placed"] = df["status"].map({
    "Placed": 1,
    "Not Placed": 0
})

df = df.drop("status", axis=1)

## Class Distribution

The target variable distribution is checked to understand whether the dataset is balanced or imbalanced before training the machine learning models.

In [252]:
print(df["status_Placed"].value_counts())

status_Placed
1    148
0     67
Name: count, dtype: int64


### Observation

The target variable contains 148 placed students and 67 not placed students. The dataset is moderately imbalanced, but the imbalance is not severe enough to require techniques such as SMOTE or random oversampling. Therefore, the original dataset is used for model training.

## Feature Selection
The input features (X) and target variable (y) are separated. The target variable is `status_Placed`, while all remaining columns are used as input features.

In [253]:
x = df.drop(["student_id", "status_Placed"], axis=1)
y = df["status_Placed"]

## Train-Test Split
The dataset is divided into training (80%) and testing (20%) sets using stratified sampling to preserve the class distribution.

In [254]:
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,stratify=y,random_state=42)

In [255]:
print("Training Features :", x_train.shape)
print("Testing Features  :", x_test.shape)
print("Training Labels   :", y_train.shape)
print("Testing Labels    :", y_test.shape)

Training Features : (172, 14)
Testing Features  : (43, 14)
Training Labels   : (172,)
Testing Labels    : (43,)


## Model Training
Four supervised machine learning classification algorithms are trained and compared to identify the best-performing model for predicting student placement.

In [256]:
scaler=StandardScaler()
X_train=scaler.fit_transform(x_train)
X_test=scaler.transform(x_test)

### Logistic Regression
Logistic Regression is used as the baseline classification model for predicting placement status.

In [257]:
model1_lr=LogisticRegression(max_iter=1000, random_state=42)
model1_lr.fit(X_train,y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,42
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


### Decision Tree
Decision Tree learns decision rules from the training data using a tree structure.

In [258]:
model2_dt=DecisionTreeClassifier(criterion='gini',
                    max_depth=5,
                    min_samples_split=5,
                    min_samples_leaf=2,
                    random_state=42  
                              )
model2_dt.fit(x_train,y_train)

,criterion,'gini'
,splitter,'best'
,max_depth,5
,min_samples_split,5
,min_samples_leaf,2
,min_weight_fraction_leaf,0.0
,max_features,None
,random_state,42
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,class_weight,None


### Random Forest
Random Forest combines multiple decision trees to improve prediction accuracy and reduce overfitting.

In [259]:
model3_rfc=RandomForestClassifier(
    n_estimators=100,
    max_depth=5,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42
)
model3_rfc.fit(x_train,y_train)

,n_estimators,100
,criterion,'gini'
,max_depth,5
,min_samples_split,5
,min_samples_leaf,2
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


### Support Vector Machine (SVM)
Support Vector Machine finds the optimal decision boundary to separate the classes.

In [260]:
model4_svc= SVC(
    kernel='rbf',
    C=1,
    gamma='scale',
    random_state=42
)
model4_svc.fit(X_train,y_train)

,C,1
,kernel,'rbf'
,degree,3
,gamma,'scale'
,coef0,0.0
,shrinking,True
,probability,False
,tol,0.001
,cache_size,200
,class_weight,None
,verbose,False


## Model Prediction

The trained models are used to predict the placement status for the test dataset.

In [261]:
y_predict_lr=model1_lr.predict(X_test)
y_predict_dt=model2_dt.predict(x_test)
y_predict_rfc=model3_rfc.predict(x_test)
y_predict_svc=model4_svc.predict(X_test)

## Model Evaluation

The models are evaluated using Accuracy, Precision, Recall, F1-Score, Confusion Matrix, Classification Report, and ROC-AUC Score.

In [262]:
print(f"Logistic Regression : {accuracy_score(y_test, y_predict_lr)*100:.2f}%")
print(f"Decision Tree       : {accuracy_score(y_test, y_predict_dt)*100:.2f}%")
print(f"Random Forest       : {accuracy_score(y_test, y_predict_rfc)*100:.2f}%")
print(f"SVM                 : {accuracy_score(y_test, y_predict_svc)*100:.2f}%")

Logistic Regression : 86.05%
Decision Tree       : 69.77%
Random Forest       : 86.05%
SVM                 : 83.72%


### Precision, Recall and F1-Score

In [263]:
print("========== Logistic Regression ==========")
print(f"Precision : {precision_score(y_test, y_predict_lr):.4f}")
print(f"Recall    : {recall_score(y_test, y_predict_lr):.4f}")
print(f"F1-Score  : {f1_score(y_test, y_predict_lr):.4f}")

print("\n========== Decision Tree ==========")
print(f"Precision : {precision_score(y_test, y_predict_dt):.4f}")
print(f"Recall    : {recall_score(y_test, y_predict_dt):.4f}")
print(f"F1-Score  : {f1_score(y_test, y_predict_dt):.4f}")

print("\n========== Random Forest ==========")
print(f"Precision : {precision_score(y_test, y_predict_rfc):.4f}")
print(f"Recall    : {recall_score(y_test, y_predict_rfc):.4f}")
print(f"F1-Score  : {f1_score(y_test, y_predict_rfc):.4f}")

print("\n========== Support Vector Machine ==========")
print(f"Precision : {precision_score(y_test, y_predict_svc):.4f}")
print(f"Recall    : {recall_score(y_test, y_predict_svc):.4f}")
print(f"F1-Score  : {f1_score(y_test, y_predict_svc):.4f}")

========== Logistic Regression ==========
Precision : 0.9286
Recall    : 0.8667
F1-Score  : 0.8966

========== Decision Tree ==========
Precision : 0.7429
Recall    : 0.8667
F1-Score  : 0.8000

========== Random Forest ==========
Precision : 0.8750
Recall    : 0.9333
F1-Score  : 0.9032

========== Support Vector Machine ==========
Precision : 0.9259
Recall    : 0.8333
F1-Score  : 0.8772


### Confusion Matrix
The confusion matrix is used to evaluate the classification performance by comparing the actual and predicted class labels.

In [264]:
print("========== Logistic Regression ==========")
print(confusion_matrix(y_test, y_predict_lr))

print("\n========== Decision Tree ==========")
print(confusion_matrix(y_test, y_predict_dt))

print("\n========== Random Forest ==========")
print(confusion_matrix(y_test, y_predict_rfc))

print("\n========== Support Vector Machine ==========")
print(confusion_matrix(y_test, y_predict_svc))

========== Logistic Regression ==========
[[11  2]
 [ 4 26]]

========== Decision Tree ==========
[[ 4  9]
 [ 4 26]]

========== Random Forest ==========
[[ 9  4]
 [ 2 28]]

========== Support Vector Machine ==========
[[11  2]
 [ 5 25]]


### ROC-AUC Score

The ROC-AUC score measures the model's ability to distinguish between the placed and not placed classes. A higher ROC-AUC score indicates better classification performance.

In [265]:
print("Logistic Regression :", roc_auc_score(y_test, y_predict_lr))
print("Random Forest       :", roc_auc_score(y_test, y_predict_rfc))

Logistic Regression : 0.8564102564102564
Random Forest       : 0.8128205128205128


### Classification Report

The classification report provides Precision, Recall, F1-Score, and Support for each class, offering a detailed evaluation of model performance.

In [266]:
print("Logistic Regression")
print(classification_report(y_test, y_predict_lr))

print("Decision Tree")
print(classification_report(y_test, y_predict_dt))

print("Random Forest")
print(classification_report(y_test, y_predict_rfc))

print("SVM")
print(classification_report(y_test, y_predict_svc))

Logistic Regression
              precision    recall  f1-score   support

           0       0.73      0.85      0.79        13
           1       0.93      0.87      0.90        30

    accuracy                           0.86        43
   macro avg       0.83      0.86      0.84        43
weighted avg       0.87      0.86      0.86        43

Decision Tree
              precision    recall  f1-score   support

           0       0.50      0.31      0.38        13
           1       0.74      0.87      0.80        30

    accuracy                           0.70        43
   macro avg       0.62      0.59      0.59        43
weighted avg       0.67      0.70      0.67        43

Random Forest
              precision    recall  f1-score   support

           0       0.82      0.69      0.75        13
           1       0.88      0.93      0.90        30

    accuracy                           0.86        43
   macro avg       0.85      0.81      0.83        43
weighted avg       0.86   

## Model Comparison

In [267]:
results = pd.DataFrame({
    "Model": ["Logistic Regression", "Decision Tree", "Random Forest", "SVM"],
    "Accuracy": [
        accuracy_score(y_test, y_predict_lr),
        accuracy_score(y_test, y_predict_dt),
        accuracy_score(y_test, y_predict_rfc),
        accuracy_score(y_test, y_predict_svc)
    ],
    "Precision": [
        precision_score(y_test, y_predict_lr),
        precision_score(y_test, y_predict_dt),
        precision_score(y_test, y_predict_rfc),
        precision_score(y_test, y_predict_svc)
    ],
    "Recall": [
        recall_score(y_test, y_predict_lr),
        recall_score(y_test, y_predict_dt),
        recall_score(y_test, y_predict_rfc),
        recall_score(y_test, y_predict_svc)
    ],
    "F1-Score": [
        f1_score(y_test, y_predict_lr),
        f1_score(y_test, y_predict_dt),
        f1_score(y_test, y_predict_rfc),
        f1_score(y_test, y_predict_svc)
    ],
    "ROC-AUC": [
        roc_auc_score(y_test, y_predict_lr),
        roc_auc_score(y_test, y_predict_dt),
        roc_auc_score(y_test, y_predict_rfc),
        roc_auc_score(y_test, y_predict_svc)
    ]
})
results=results.round(4)
print(results)

                 Model  Accuracy  Precision  Recall  F1-Score  ROC-AUC
0  Logistic Regression    0.8605     0.9286  0.8667    0.8966   0.8564
1        Decision Tree    0.6977     0.7429  0.8667    0.8000   0.5872
2        Random Forest    0.8605     0.8750  0.9333    0.9032   0.8128
3                  SVM    0.8372     0.9259  0.8333    0.8772   0.8397


## Best Model Selection

Four supervised machine learning models were trained and evaluated using Accuracy, Precision, Recall, F1-Score, and ROC-AUC.

Although Logistic Regression and Random Forest achieved the same accuracy (86.05%), Logistic Regression achieved higher Precision (0.9286) and a higher ROC-AUC score (0.8564). Since the dataset contains only 215 records, Logistic Regression was selected as the final deployment model because it provides strong predictive performance while remaining simple, interpretable, and less prone to overfitting.

## Save Model

In [268]:
os.makedirs("model", exist_ok=True)
joblib.dump(model1_lr, "model/student_placement_logistic_model.pkl")
joblib.dump(scaler, "model/scaler.pkl")
joblib.dump(encoder, "model/encoder.pkl")
joblib.dump(x.columns.tolist(), "model/feature_names.pkl")
joblib.dump(scaler, "model/scaler.pkl")
joblib.dump(x.columns.tolist(), "model/feature_names.pkl")

['model/feature_names.pkl']

## Conclusion

This project developed a supervised machine learning model to predict student placement based on academic performance and employability-related features.

Four classification algorithms were trained and evaluated. Logistic Regression was selected as the final model because it achieved competitive accuracy (86.05%), the highest precision (0.9286), and the highest ROC-AUC score (0.8564), while remaining simple and interpretable for deployment.

The trained model has been saved and is ready to be integrated into a web application for real-time placement prediction.